# Getting Started with the Anthropic Python SDK

This notebook follows the course's **"Making a request"** lesson step by step:

1. Install the Anthropic Python SDK and `python-dotenv`
2. Store your API key in a `.env` file and load it
3. Create a client (and pick a model)
4. Make a request and extract the response

> **Before you run anything:** make sure the kernel (top-right) is **Python (anthropic-cert)**, and that you've copied `.env.example` to `.env` and pasted your real key into it.

## Step 1 — Install the SDK and python-dotenv

These are already installed in this environment, but this is the command the course shows. The leading `%pip` runs it inside *this* notebook's kernel.

In [ ]:
%pip install anthropic python-dotenv

## Step 2 — Load your API key from `.env`

Your `.env` file holds the key like this (already set up in `.env.example`):

```
ANTHROPIC_API_KEY="your-api-key-here"
```

`load_dotenv()` reads that file and puts its values into environment variables. This keeps your key out of your code so you never accidentally commit it (`.env` is in `.gitignore`).

The check below confirms the key loaded **without printing it**.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("ANTHROPIC_API_KEY")

if not api_key:
    print("No key found. Copy .env.example to .env and paste your key, then re-run this cell.")
elif not api_key.startswith("sk-ant-"):
    print("A value loaded, but it doesn't look like an Anthropic key (should start with 'sk-ant-').")
else:
    print(f"Key loaded OK (starts with {api_key[:10]}..., length {len(api_key)}).")

## Step 3 — Create the client and pick a model

The `Anthropic()` client automatically picks up `ANTHROPIC_API_KEY` from the environment, so you don't pass the key in explicitly.

We store the model name in a `model` variable — the course reuses this variable in later lessons, so it's a handy habit. The course's `claude-sonnet-4-0` is now deprecated and returns a 404, so we use **`claude-sonnet-5`** (the current balanced model). Other options: `claude-haiku-4-5-20251001` (fastest/cheapest) or `claude-opus-4-8` (most capable).

In [ ]:
from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-5"

print(f"Client ready. Model: {model}")

## Step 4 — Make a request and extract the response

`client.messages.create()` takes three key parameters:

- **`model`** — which Claude model to use
- **`max_tokens`** — a *safety limit* on response length, not a target. Claude writes what it thinks is appropriate and stops; it only cuts off if it hits this ceiling.
- **`messages`** — the conversation so far. Each message is a dict with a `role` (`"user"` for what you send, `"assistant"` for Claude's replies) and `content` (the text).

In [ ]:
message = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=[
        {
            "role": "user",
            "content": "What is quantum computing? Answer in one sentence"
        }
    ]
)

# The response object holds a lot of info; usually you just want the text:
message.content[0].text

### Examine the full response object

Beyond the text, the response carries the model used, why it stopped, and token usage. Worth a look:

In [ ]:
print("Model:        ", message.model)
print("Stop reason:  ", message.stop_reason)
print("Input tokens: ", message.usage.input_tokens)
print("Output tokens:", message.usage.output_tokens)
print()
print("Raw object:")
message